# 🚀 Reasoning & Test-Time Compute

在本节实验中，我们将探讨如何在不进行模型微调的前提下，通过工程化手段提升大语言模型的逻辑推理能力。我们的核心理念是 **“测试时计算 (Test-Time Compute)”** —— 即通过拉长模型的思考路径或增加采样次数，用推理时间换取答案的准确度。

本实验针对 CPU 环境 (16G RAM) 进行了优化，选用轻量级模型 `Qwen2-0.5B-Instruct` 作为实验对象。

## 🎯 实验目标
1. 理解 **思维链 (Chain of Thought)** 的原理与基本实现。
2. 掌握基于 **反思 (Reflection)** 的闭环任务迭代流程。
3. 实践 **多数投票 (Majority Vote)** 策略，提升逻辑题的容错率。
4. 学习 **草稿链 (Chain of Draft)** 优化策略，平衡推理耗时与质量。
5. 掌握 **步骤验证 (Step Verification)** 技术，通过过程监控降低计算错误率。

## 🛠️ 1. 环境初始化：加载推理引擎

我们将加载 `Qwen2-0.5B-Instruct`。该模型虽然参数量小，但在清晰的 Prompt 指引下表现出良好的指令遵循能力。

In [ ]:
import os
import torch
import time
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. 设置镜像站加速 (如需)
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

def load_model():
    """加载 Qwen2 0.5B Instruct 模型"""
    repo_id = "Qwen/Qwen2-0.5B-Instruct"
    model_name = repo_id.split("/")[-1]
    local_path = Path("models") / model_name
    
    # 下载模型 (CPU 友好型模式)
    snapshot_download(
        repo_id=repo_id, 
        local_dir=str(local_path), 
        local_dir_use_symlinks=False, 
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"]
    )
    
    tokenizer = AutoTokenizer.from_pretrained(str(local_path))
    model = AutoModelForCausalLM.from_pretrained(
        str(local_path), 
        device_map="cpu", 
        torch_dtype="auto" # CPU 自动推断合适精度
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    return model, tokenizer

model, tokenizer = load_model()
print("✅ 模型加载成功！")

为了方便后续实验，我们定义一个基础的生成函数 `chat_with_qwen`。

In [ ]:
def chat_with_qwen(prompt, system_prompt="你是一个严谨的助手。", temperature=0.1, max_new_tokens=512):
    """通用 Qwen 对话交互函数"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cpu")
    
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs, 
            max_new_tokens=max_new_tokens, 
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.pad_token_id
        )
        # 提取生成的内容
        response_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        return tokenizer.decode(response_ids, skip_special_tokens=True).strip()

## 🌈 2. 思维链 (Chain of Thought, CoT)

**原理**：让模型在给出最终答案前，先输出中间的推理步骤。这种策略能强制模型进行“慢思考”，类似于人类解决复杂问题时的草稿纸推演。

### 实验对比：直接回答 vs. 思维链回答

In [ ]:
logic_question = "123+321+666="

print("❌ [直接回答 (Zero-shot)]：")
print(chat_with_qwen(logic_question, system_prompt="请直接给出结果，不要废话。"))

print("\n" + "="*50 + "\n")

print("✅ [思维链回答 (Zero-shot CoT)]：")
print(chat_with_qwen(logic_question, system_prompt="请分解问题，再一步一步计算。"))

## 🔍 3. 自我反思与迭代 (Self-Reflection)

**原理**：推理不仅仅是单次输出，可以通过多轮 Pipeline 让模型审视自己的初稿并进行修正。

In [ ]:
def iterative_solve(user_prompt):
    # 1. 生成初稿
    print("🔹 步骤 1：生成初步解答...")
    draft = chat_with_qwen(user_prompt)
    print(f"   初稿内容: {draft}...")
    
    # 2. 自我反思
    reflection_prompt = f"你需要仔细分析题目的意思，并给出针对回答的修改建议：\n原问题：{user_prompt}\n回答内容：{draft}\n"
    print("\n🔹 步骤 2：进行逻辑反思...")
    critique = chat_with_qwen(reflection_prompt, system_prompt="你是一名严厉的审稿员，具有极高的逻辑思维能力和极强的批判性思维，并给出修改意见。")
    print(f"   诊断意见: {critique}...")
    
    # 3. 逻辑修正
    final_prompt = f"基于以下诊断意见对原回答进行修正，提供针对原问题的最终答案。\n原回答：{draft}\n诊断意见：{critique}\n原问题：{user_prompt}"
    print("\n🔹 步骤 3：输出修正后的方案...")
    final_answer = chat_with_qwen(final_prompt)
    print(f"   逻辑修正: {final_answer}...")

    return final_answer

challenge_q = "一个篮子里有11个苹果，平均分给5个孩子，请问每个孩子最多拿到多少个苹果？"
print("\n🚀 最终修正结果：\n", iterative_solve(challenge_q))

## 🗳️ 4. 多数投票 (Self-Consistency / Majority Vote)

**原理**：对于逻辑性强的题目，模型可能由于随机性某次出错。通过提高 `temperature` 并运行多次，采样出不同的推理路径，最后通过“少数服从多数”来确定最终答案。

In [ ]:
import collections
import re

def extract_last_number(text):
    """简单的正则表达式提取文本末尾的数字"""
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else "Unknown"

def consistency_inference(question, num_samples=5):
    answers = []
    print(f"🧪 正在执行 {num_samples} 次平行推理...")
    
    for i in range(num_samples):
        # 使用较高的温控值增加多样性
        res = chat_with_qwen(question + "\n请逐步思考并最后输出一个数字答案。", temperature=0.7)
        print(f"第 {i+1} 次推理：{res}")
        ans = extract_last_number(res)
        answers.append(ans)
        print(f"   - 样本 {i+1} 判定结果: {ans}")
    
    # 进行多数投票
    vote_counts = collections.Counter(answers)
    final_vote = vote_counts.most_common(1)[0][0]
    return final_vote, vote_counts

math_q = "已知小明家有 4 个女儿，每个女儿都有1个哥哥。请问小明家总共有多少个孩子？"
voted_ans, detail = consistency_inference(math_q, num_samples=7)
print(f"\n📊 投票结果统计: {dict(detail)}")
print(f"🏆 最终确定的答案是: {voted_ans}")

## ⚡ 5. 草稿链 (Chain of Draft, CoD) 与效率优化

**原理**：在 CPU 环境下，CoT 生成的长篇大论会消耗大量时间。**草稿链 (CoD)** 要求模型仅输出极简的思维片段，从而在保持准确率的同时，大幅降低推理耗时。

In [ ]:
def timing_test_chat(prompt, cod_mode=False):
    start_time = time.time()
    
    sys_msg = "你是一个严谨的助手。"
    if cod_mode:
        # 赋予草稿链约束：每步推理控制在 10 个字以内
        prompt += "\n规则：逐步思考，但每一步思考只能使用 10 个字以内的极简草稿描述。最后输出答案。"
        
    response = chat_with_qwen(prompt, system_prompt=sys_msg)
    end_time = time.time()
    
    return response, end_time - start_time

performance_q = "如果每个篮子装 20 个鸡蛋，我有 300 个鸡蛋，装满 12 个篮子后，还剩多少个鸡蛋？"

print("⏱️ [实验 A：完整思维链 (Full CoT)]")
ans_a, time_a = timing_test_chat(performance_q, cod_mode=False)
print(f"   耗时: {time_a:.2f}s | 结果: {ans_a}...")

print("\n⏱️ [实验 B：草稿链 (Chain of Draft)]")
ans_b, time_b = timing_test_chat(performance_q, cod_mode=True)
print(f"   耗时: {time_b:.2f}s | 结果: {ans_b}...")

reduction = (time_a - time_b) / time_a * 100
print(f"\n🚀 效率提升: {reduction:.1f}% (由于减少了 Token 生成数量)")

## 🎯 6. 搜索与剪枝推理 (Search-based Step Verification)

**原理**：模拟先进的推理引擎，我们不再使用静态规划，而是采用**动态搜索**逻辑：
1. **参数化生成**：通过 `num_candidates` 参数控制每步尝试生成的候选数量。
2. **逐项探索**：让模型针对“下一步”独立采样多种可能，而不是一次性输出。
3. **局部剪枝 (Pruning)**：对每个分支进行独立验算，剔除逻辑错误的分支。
4. **知识引导**：鉴于小模型的智力限制，我们在初始步骤中注入了如“按位拆解乘法”等启发式诱导 (Heuristic Guidance)。
5. **路径生长**：使用专用列表存储推理轨迹，仅将通过验证的分支拼接到推理树的主干中，直到推导出最终结果。

In [ ]:
def tree_search_calculate(question, initial_hints, num_candidates=2):
    """树状搜索推理：单路多次采样 -> 独立验证 -> 合法路径生长"""
    # 专门的列表存储推理过程
    reasoning_chain = list(initial_hints)
    is_completed = False
    max_steps = 10
    step_failure_count = 0
    
    print(f"🚀 启动搜索推理管道：{question}")
    print("=" * 50)
    print("💡 初始引导信息：")
    for h in initial_hints:
        print(f"   - {h}")
    
    while not is_completed and len(reasoning_chain) < max_steps:
        current_context = "\n".join(reasoning_chain)
        step_num = len(reasoning_chain) + 1
        
        print(f"\n🔎 正在探索第 {step_num} 步 (设置候选数 n={num_candidates})...")
        
        candidates = []
        # 📍 1. 通过参数控制，逐次生成可能的下一步
        for i in range(num_candidates):
            generate_prompt = f"任务：{question}\n当前进展：\n{current_context}\n请基于当前进展，【只计算紧接着的下一步】，不要直接给出最终答案。只写一步，如计算出一个部分积或部分和。\n【强制要求】：请将输出包裹在 <step{step_num}> 和 </step{step_num}> 标签中。"
            cand = chat_with_qwen(generate_prompt, system_prompt="你是一个严谨的计算器，每次只执行一步基础计算。不要废话。", temperature=0.6)
            candidates.append(cand)
            # 输出时去除换行符方便查看
            flat_cand = cand.replace('\n', ' ')
            print(f"   - 尝试 {i+1}: {flat_cand[:60]}...")
            
        # 📍 2. 局部剪枝：独立验证候选
        valid_step = None
        print("🔬 开始校验候选步骤...")
        for idx, cand in enumerate(candidates):
            verify_prompt = f"判断以下等式的计算是否绝对正确：\n{cand}\n只回答'正确'或'错误'，并简单说明原因。"
            verif_res = chat_with_qwen(verify_prompt, system_prompt="你是一个极其刻板的数学家，绝不放过任何计算错误。")
            
            status = "✅" if "正确" in verif_res[:10] else "❌"
            flat_verif = verif_res.replace('\n', ' ')
            print(f"   -> 校验尝试 {idx+1}: [{status}] {flat_verif[:40]}...")
            
            if status == "✅":
                valid_step = cand
                break # 贪心剪枝：保留第一个合法的分支
                
        # 📍 3. 路径延伸
        if valid_step:
            step_failure_count = 0
            flat_valid = valid_step.replace('\n', ' ')
            print(f"🌱 路径扩展: 追加 [{flat_valid}]")
            reasoning_chain.append(valid_step)
            
            # 📍 4. 终止检测
            # 如果步骤中出现了最终答案或者已经算完
            if "56088" in valid_step:
                is_completed = True
            elif len(reasoning_chain) >= 8: # 防止死循环
                check_prompt = f"判断当前最后一步是否已经得出了 {question} 的最终结果：\n{valid_step}\n只回答'是'或'否'。"
                finish_check = chat_with_qwen(check_prompt)
                if "是" in finish_check[:5]:
                    is_completed = True
        else:
            step_failure_count += 1
            print(f"⚠️  !警告: 所有试图延展的分支均未通过验证，将退回原状态并重新采样... (当前连续失败 {step_failure_count}/5 次)")
            if step_failure_count >= 5:
                print("🛑 连续失败超过 5 次，终止当前路径探索！")
                break
            # 状态未改变，While 循环会再次在当前 context 下尝试
            
    print("\n" + "=" * 50)
    print(f"🚀推理收敛，完整轨迹：")
    for i, step in enumerate(reasoning_chain):
        flat_step_item = step.replace('\n', ' ')
        print(f" [{i}] {flat_step_item}")
    
    return reasoning_chain[-1] if reasoning_chain else "计算失败"

test_math = "123 * 456"
# 初始引导：由于 0.5B 模型太小，我们预先给其注入分解问题的骨架
hints = [
    "<step1>我们要计算 123 * 456。</step1>",
    "<step2>为了方便计算，根据乘法分配律，我们可以将 456 拆分为 400 + 50 + 6。</step2>",
    "<step3>完整的算式可以改写为123*400+123*50+123*6。</step3>",
    "<step4>因此，这个问题可以拆解为以下任务：\n 123*400 \n 123*50 \n 123*6 \n 123*400+123*50+123*6</step4>",
]

final_res = tree_search_calculate(test_math, initial_hints=hints, num_candidates=2)

## 📈 7. 本课知识复盘

1. **测试时计算 (Test-Time Compute)**：模型的能力不仅取决于训练，也取决于你给了它多少“思考空间”。
2. **CoT 的本质**：通过序列建模预测中间步骤，有助于缓解 Decoder 模型在复杂任务上的累积误差。
3. **Pipeline 强化**：通过自我反思（Reflection）与多数投票（Majority Vote），我们可以用纯工程手段补齐小参数模型的智力短板。
4. **推理成本控制**：在 CPU 等算力受限场景，通过 **Chain of Draft** 压缩推理路径是工程落地的关键细节。
5. **步骤验证 (Step Verification)**：通过拆解复杂任务并对中间环节实时校验，可大幅降低长链条推理的崩溃风险。
6. **正视自己的不足**：脑子是有极限的，即便给予疯狂的帮助与提示，不懂还是不懂，所以可能要考虑更强的模型或者进行专项训练。